# LDA: Topic Term Drift (TTD)

**Scenario 3a** — Three complementary drift metrics:
1. **Endpoint TTD**: cosine distance from first to last year (total drift)
2. **Trajectory from baseline**: cosine_sim(t₀, t) per year (when drift happens)
3. **Year-over-year drift**: cosine_sim(t, t+1) per year (smoothness of evolution)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/lda/temporal")
RESULT_DIR = Path("../../../../results/lda/consistency")

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Reading from: {TEMPORAL_DIR}")
print(f"Saving to: {RESULT_DIR}")

Reading from: ../../../../results/lda/temporal
Saving to: ../../../../results/lda/consistency


In [3]:
def parse_words(words_str):
    return [w.strip() for w in str(words_str).split(",")]


def words_to_vector(words, vocab_index):
    vec = np.zeros(len(vocab_index))
    for w in words:
        if w in vocab_index:
            vec[vocab_index[w]] = 1.0
    return vec


def cos_sim(v1, v2):
    """Cosine similarity between two vectors."""
    return float(cosine_similarity(v1.reshape(1, -1), v2.reshape(1, -1))[0, 0])

## 1. Endpoint TTD
Total drift: `TTD = 1 - cosine_sim(words_first, words_last)`

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Endpoint TTD: {subject.upper()} (LDA)")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_words[key] = parse_words(row["top_words"])

    years = sorted(evo_df["year"].unique())
    all_topics = sorted(evo_df["topic_id"].unique())

    # Build global vocabulary
    all_w = set()
    for ws in topic_words.values():
        all_w.update(ws)
    vocab_index = {w: i for i, w in enumerate(sorted(all_w))}

    rows = []
    for tid in all_topics:
        topic_years = sorted([y for y in years if (y, tid) in topic_words])
        if len(topic_years) < 2:
            continue

        t_first, t_last = topic_years[0], topic_years[-1]
        v_first = words_to_vector(topic_words[(t_first, tid)], vocab_index)
        v_last = words_to_vector(topic_words[(t_last, tid)], vocab_index)
        sim = cos_sim(v_first, v_last)

        rows.append({
            "subject": subject, "topic_id": tid,
            "first_year": t_first, "last_year": t_last,
            "n_years": len(topic_years),
            "cosine_sim": round(sim, 6),
            "ttd": round(1.0 - sim, 6),
            "words_first": ", ".join(topic_words[(t_first, tid)]),
            "words_last": ", ".join(topic_words[(t_last, tid)]),
        })

    df = pd.DataFrame(rows)
    df.to_csv(RESULT_DIR / subject / "topic_term_drift.csv", index=False)

    summary = {"subject": subject, "n_topics": len(df),
               "ttd_mean": round(df["ttd"].mean(), 6),
               "ttd_std": round(df["ttd"].std(), 6),
               "ttd_median": round(df["ttd"].median(), 6),
               "pct_stable": round((df["ttd"] < 0.5).mean() * 100, 2),
               "pct_drifted": round((df["ttd"] >= 0.5).mean() * 100, 2)}
    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "ttd_summary.csv", index=False)

    print(f"  TTD: mean={summary['ttd_mean']:.4f} ± {summary['ttd_std']:.4f}, "
          f"median={summary['ttd_median']:.4f}")
    print(f"  Stable (TTD<0.5): {summary['pct_stable']:.1f}%  |  "
          f"Drifted (TTD≥0.5): {summary['pct_drifted']:.1f}%")

    print(f"\n  �� Top 5 most DRIFTED:")
    for _, r in df.nlargest(5, 'ttd').iterrows():
        print(f"    T{int(r['topic_id']):>3} | TTD={r['ttd']:.4f} | "
              f"{r['words_first'][:40]}... → {r['words_last'][:40]}...")

    print(f"  🔒 Top 5 most STABLE:")
    for _, r in df.nsmallest(5, 'ttd').iterrows():
        print(f"    T{int(r['topic_id']):>3} | TTD={r['ttd']:.4f} | {r['words_first'][:60]}")


Endpoint TTD: CS (LDA)
  TTD: mean=0.9041 ± 0.1103, median=0.9000
  Stable (TTD<0.5): 0.0%  |  Drifted (TTD≥0.5): 100.0%

  �� Top 5 most DRIFTED:
    T  0 | TTD=1.0000 | hair, photorealistic, moebius, interpola... → image, diffusion, images, editing, model...
    T  6 | TTD=1.0000 | huffman, unequal, packing, sphere, equip... → rdp, distortion, perception, bernoulli, ...
    T 10 | TTD=1.0000 | cardiac, surgery, surgeon, dexterity, st... → mri, clinical, segmentation, ct, imaging...
    T 12 | TTD=1.0000 | bidder, bids, bidding, bidders, algorith... → regret, online, optimal, bandits, algori...
    T 13 | TTD=1.0000 | fix, distributive, endo, points, calcula... → textsc, stv, elections, voting, seat, rl...
  🔒 Top 5 most STABLE:
    T 61 | TTD=0.5000 | problem, csp, algorithm, problems, time, algorithms, aixi, m
    T  7 | TTD=0.6000 | logic, programs, termination, semantics, revision, logics, p
    T 53 | TTD=0.6000 | exploration, learning, grep, policy, observable, phenotype, 
    

## 2. Drift Trajectory from Baseline
For each topic: `cosine_sim(words_t₀, words_t)` per year — shows **when** drift happens

In [5]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Drift Trajectory: {subject.upper()} (LDA)")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_words[key] = parse_words(row["top_words"])

    years = sorted(evo_df["year"].unique())
    all_topics = sorted(evo_df["topic_id"].unique())

    all_w = set()
    for ws in topic_words.values():
        all_w.update(ws)
    vocab_index = {w: i for i, w in enumerate(sorted(all_w))}

    traj_rows = []
    for tid in all_topics:
        topic_years = sorted([y for y in years if (y, tid) in topic_words])
        if len(topic_years) < 2:
            continue

        t0 = topic_years[0]
        v_baseline = words_to_vector(topic_words[(t0, tid)], vocab_index)

        for y in topic_years:
            v_y = words_to_vector(topic_words[(y, tid)], vocab_index)
            sim = cos_sim(v_baseline, v_y)
            traj_rows.append({
                "subject": subject, "topic_id": tid,
                "baseline_year": t0, "year": y,
                "years_elapsed": y - t0,
                "cosine_sim_to_baseline": round(sim, 6),
                "drift_from_baseline": round(1.0 - sim, 6),
            })

    traj_df = pd.DataFrame(traj_rows)
    traj_df.to_csv(RESULT_DIR / subject / "ttd_trajectory.csv", index=False)

    # Average drift trajectory across all topics per year
    avg_traj = traj_df.groupby("year").agg(
        avg_sim=("cosine_sim_to_baseline", "mean"),
        avg_drift=("drift_from_baseline", "mean"),
        n_topics=("topic_id", "count")
    ).reset_index()
    avg_traj.insert(0, "subject", subject)
    avg_traj.to_csv(RESULT_DIR / subject / "ttd_trajectory_avg.csv", index=False)

    print(f"  Avg drift from baseline per year:")
    for _, r in avg_traj.iterrows():
        bar = '█' * int(r['avg_drift'] * 50)
        print(f"    {int(r['year'])}: sim={r['avg_sim']:.4f}  drift={r['avg_drift']:.4f}  {bar}")
    print(f"  Saved: ttd_trajectory.csv, ttd_trajectory_avg.csv")


Drift Trajectory: CS (LDA)
  Avg drift from baseline per year:
    2000: sim=1.0000  drift=0.0000  
    2001: sim=0.3412  drift=0.6588  ████████████████████████████████
    2002: sim=0.1482  drift=0.8518  ██████████████████████████████████████████
    2003: sim=0.1296  drift=0.8704  ███████████████████████████████████████████
    2004: sim=0.1403  drift=0.8597  ██████████████████████████████████████████
    2005: sim=0.0870  drift=0.9130  █████████████████████████████████████████████
    2006: sim=0.1127  drift=0.8873  ████████████████████████████████████████████
    2007: sim=0.0914  drift=0.9086  █████████████████████████████████████████████
    2008: sim=0.0839  drift=0.9161  █████████████████████████████████████████████
    2009: sim=0.0966  drift=0.9034  █████████████████████████████████████████████
    2010: sim=0.0841  drift=0.9159  █████████████████████████████████████████████
    2011: sim=0.1176  drift=0.8824  ████████████████████████████████████████████
    2012: sim=0.1076

## 3. Year-over-Year Drift
`cosine_sim(words_t, words_t+1)` — detects **sudden jumps** vs gradual evolution

In [6]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Year-over-Year Drift: {subject.upper()}")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row['year']), int(row['topic_id']))
        topic_words[key] = parse_words(row['top_words'])

    years = sorted(evo_df['year'].unique())

    # Build global vocabulary
    all_w = set()
    for ws in topic_words.values():
        all_w.update(ws)
    vocab_index = {w: i for i, w in enumerate(sorted(all_w))}

    yoy_rows = []

    # Iterate by CONSECUTIVE CALENDAR YEARS
    for i in range(len(years) - 1):
        t, t1 = int(years[i]), int(years[i + 1])

        # Find topics present in BOTH years
        topics_t = {tid for (y, tid) in topic_words if y == t}
        topics_t1 = {tid for (y, tid) in topic_words if y == t1}
        common_topics = sorted(topics_t & topics_t1)

        for tid in common_topics:
            v_t = words_to_vector(topic_words[(t, tid)], vocab_index)
            v_t1 = words_to_vector(topic_words[(t1, tid)], vocab_index)
            sim = cos_sim(v_t, v_t1)

            yoy_rows.append({
                'subject': subject, 'topic_id': tid,
                'year_from': t, 'year_to': t1,
                'cosine_sim': round(sim, 6),
                'drift': round(1.0 - sim, 6),
            })

    yoy_df = pd.DataFrame(yoy_rows)
    yoy_df.to_csv(RESULT_DIR / subject / "ttd_year_over_year.csv", index=False)

    # Per-topic summary
    topic_yoy = yoy_df.groupby('topic_id').agg(
        avg_yoy_sim=('cosine_sim', 'mean'),
        min_yoy_sim=('cosine_sim', 'min'),
        avg_yoy_drift=('drift', 'mean'),
        max_single_jump=('drift', 'max'),
    ).reset_index()
    topic_yoy.insert(0, 'subject', subject)
    topic_yoy.to_csv(RESULT_DIR / subject / "ttd_yoy_per_topic.csv", index=False)

    # Avg across all topics per year transition
    avg_yoy = yoy_df.groupby(['year_from', 'year_to']).agg(
        avg_sim=('cosine_sim', 'mean'),
        avg_drift=('drift', 'mean'),
        n_topics=('topic_id', 'count')
    ).reset_index()
    avg_yoy.insert(0, 'subject', subject)
    avg_yoy.to_csv(RESULT_DIR / subject / "ttd_yoy_avg.csv", index=False)

    print(f"  Avg year-over-year drift (consecutive calendar years):")
    for _, r in avg_yoy.iterrows():
        bar = chr(9608) * int(r['avg_drift'] * 50)
        print(f"    {int(r['year_from'])}→{int(r['year_to'])}: sim={r['avg_sim']:.4f}  "
              f"drift={r['avg_drift']:.4f}  ({int(r['n_topics'])} topics)  {bar}")

    # Top 5 biggest single-year jumps
    print(f"\n  Top 5 biggest single-year JUMPS:")
    for _, r in yoy_df.nlargest(5, 'drift').iterrows():
        print(f"    T{int(r['topic_id']):>3} | {int(r['year_from'])}→{int(r['year_to'])} | "
              f"drift={r['drift']:.4f}")

    overall_sim = yoy_df['cosine_sim'].mean()
    overall_drift = yoy_df['drift'].mean()
    print(f"\n  Overall: avg_yoy_sim={overall_sim:.4f}  avg_yoy_drift={overall_drift:.4f}")
    print(f"  Saved: ttd_year_over_year.csv, ttd_yoy_per_topic.csv, ttd_yoy_avg.csv")


Year-over-Year Drift: CS
  Avg year-over-year drift (consecutive calendar years):
    2000→2001: sim=0.1385  drift=0.8615  (39 topics)  ███████████████████████████████████████████
    2001→2002: sim=0.1383  drift=0.8617  (47 topics)  ███████████████████████████████████████████
    2002→2003: sim=0.1327  drift=0.8673  (49 topics)  ███████████████████████████████████████████
    2003→2004: sim=0.1353  drift=0.8647  (51 topics)  ███████████████████████████████████████████
    2004→2005: sim=0.1660  drift=0.8340  (53 topics)  █████████████████████████████████████████
    2005→2006: sim=0.1774  drift=0.8226  (53 topics)  █████████████████████████████████████████
    2006→2007: sim=0.1491  drift=0.8509  (55 topics)  ██████████████████████████████████████████
    2007→2008: sim=0.1778  drift=0.8222  (54 topics)  █████████████████████████████████████████
    2008→2009: sim=0.1464  drift=0.8536  (56 topics)  ██████████████████████████████████████████
    2009→2010: sim=0.1873  drift=0.8127  (5

## Inspect Individual Topics
Two helper functions to drill into specific topics.

In [7]:
def show_topic_yoy(topic_id, subject):
    """
    Show year-over-year drift for a specific topic with drift labels.
    Labels: Stable (<0.3), Moderate (0.3-0.6), Aggressive (0.6-0.8), Extreme (>0.8)
    """
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    topic_evo = evo_df[evo_df["topic_id"] == topic_id].sort_values("year")

    if len(topic_evo) < 2:
        print(f"Topic {topic_id} has <2 years of data in {subject}")
        return

    # Build vocabulary
    all_words_set = set()
    tw = {}
    for _, row in topic_evo.iterrows():
        words = parse_words(row["top_words"])
        tw[int(row["year"])] = words
        all_words_set.update(words)
    vi = {w: i for i, w in enumerate(sorted(all_words_set))}

    topic_years = sorted(tw.keys())

    print(f"\n{'='*70}")
    print(f"YoY Drift: Topic {topic_id} — {subject.upper()}")
    print(f"Active: {topic_years[0]}–{topic_years[-1]} ({len(topic_years)} years)")
    print(f"{'='*70}")

    drifts = []
    for i in range(len(topic_years) - 1):
        t, t1 = topic_years[i], topic_years[i + 1]
        v_t = words_to_vector(tw[t], vi)
        v_t1 = words_to_vector(tw[t1], vi)
        sim = cos_sim(v_t, v_t1)
        drift = 1.0 - sim
        drifts.append(drift)

        # Label
        if drift < 0.3:
            label = "🟢 Stable"
        elif drift < 0.6:
            label = "🟡 Moderate"
        elif drift < 0.8:
            label = "�� Aggressive"
        else:
            label = "🔴 Extreme"

        bar = chr(9608) * int(drift * 30)
        print(f"  {t}→{t1}: sim={sim:.4f}  drift={drift:.4f}  {bar}  {label}")

        # Show word changes
        added = set(tw[t1]) - set(tw[t])
        removed = set(tw[t]) - set(tw[t1])
        if added or removed:
            if removed:
                print(f"           − {', '.join(sorted(removed))}")
            if added:
                print(f"           + {', '.join(sorted(added))}")

    # Summary
    avg_d = np.mean(drifts)
    max_d = np.max(drifts)
    if avg_d < 0.3:
        verdict = "🟢 STABLE — vocabulary is consistent over time"
    elif avg_d < 0.5:
        verdict = "🟡 MODERATE — topic evolves gradually"
    elif avg_d < 0.7:
        verdict = "🟠 DRIFTING — significant vocabulary changes"
    else:
        verdict = "🔴 HIGHLY UNSTABLE — topic vocabulary reshuffles frequently"

    print(f"\n  Avg drift: {avg_d:.4f} | Max jump: {max_d:.4f}")
    print(f"  Verdict: {verdict}")


def show_topic_endpoint(topic_id, subject):
    """
    Show endpoint drift for a specific topic: first year vs last year.
    """
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    topic_evo = evo_df[evo_df["topic_id"] == topic_id].sort_values("year")

    if len(topic_evo) < 2:
        print(f"Topic {topic_id} has <2 years of data in {subject}")
        return

    # Build vocabulary
    tw = {}
    all_words_set = set()
    for _, row in topic_evo.iterrows():
        words = parse_words(row["top_words"])
        tw[int(row["year"])] = words
        all_words_set.update(words)
    vi = {w: i for i, w in enumerate(sorted(all_words_set))}

    topic_years = sorted(tw.keys())
    t_first, t_last = topic_years[0], topic_years[-1]

    v_first = words_to_vector(tw[t_first], vi)
    v_last = words_to_vector(tw[t_last], vi)
    sim = cos_sim(v_first, v_last)
    ttd = 1.0 - sim

    # Words analysis
    w_first = set(tw[t_first])
    w_last = set(tw[t_last])
    kept = sorted(w_first & w_last)
    removed = sorted(w_first - w_last)
    added = sorted(w_last - w_first)

    if ttd < 0.3:
        verdict = "🟢 STABLE — topic maintained its core vocabulary"
    elif ttd < 0.5:
        verdict = "🟡 MODERATE — topic evolved but recognizable"
    elif ttd < 0.7:
        verdict = "🟠 SIGNIFICANT DRIFT — topic substantially changed"
    else:
        verdict = "🔴 COMPLETE DRIFT — topic vocabulary almost entirely replaced"

    print(f"\n{'='*70}")
    print(f"Endpoint Drift: Topic {topic_id} — {subject.upper()}")
    print(f"{'='*70}")
    print(f"  Active: {t_first}–{t_last} ({len(topic_years)} years)")
    print(f"  Cosine similarity: {sim:.4f}")
    print(f"  TTD (drift):       {ttd:.4f}")
    print(f"  Verdict: {verdict}")
    print(f"\n  First year ({t_first}): {', '.join(tw[t_first])}")
    print(f"  Last year  ({t_last}): {', '.join(tw[t_last])}")
    print(f"\n  🔒 Kept ({len(kept)}):    {', '.join(kept) if kept else '(none)'}")
    print(f"  ➖ Removed ({len(removed)}): {', '.join(removed) if removed else '(none)'}")
    print(f"  ➕ Added ({len(added)}):   {', '.join(added) if added else '(none)'}")


def show_topic_trajectory(topic_id, subject):
    """
    Show drift trajectory from baseline (first year) for a specific topic.
    cosine_sim(words_t0, words_t) per year — shows when drift accelerates.
    """
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    topic_evo = evo_df[evo_df["topic_id"] == topic_id].sort_values("year")

    if len(topic_evo) < 2:
        print(f"Topic {topic_id} has <2 years of data in {subject}")
        return

    tw = {}
    all_words_set = set()
    for _, row in topic_evo.iterrows():
        words = parse_words(row["top_words"])
        tw[int(row["year"])] = words
        all_words_set.update(words)
    vi = {w: i for i, w in enumerate(sorted(all_words_set))}

    topic_years = sorted(tw.keys())
    t0 = topic_years[0]
    v_baseline = words_to_vector(tw[t0], vi)

    print(f"\n{'='*70}")
    print(f"Drift Trajectory: Topic {topic_id} — {subject.upper()}")
    print(f"Baseline: {t0} → {', '.join(tw[t0])}")
    print(f"{'='*70}")

    prev_drift = 0.0
    for y in topic_years:
        v_y = words_to_vector(tw[y], vi)
        sim = cos_sim(v_baseline, v_y)
        drift = 1.0 - sim

        # Delta from previous year
        delta = drift - prev_drift
        if delta > 0.1:
            delta_label = f" ↑{delta:+.3f} ⚡"
        elif delta > 0.02:
            delta_label = f" ↑{delta:+.3f}"
        elif delta < -0.02:
            delta_label = f" ↓{delta:+.3f} 🔄"
        else:
            delta_label = f"  {delta:+.3f} —"
        prev_drift = drift

        # Shared words with baseline
        shared = sorted(set(tw[t0]) & set(tw[y]))
        n_shared = len(shared)
        n_total = len(tw[t0])

        bar = chr(9608) * int(drift * 30)
        print(f"  {y}: sim={sim:.4f}  drift={drift:.4f}  {bar}{delta_label}  "
              f"({n_shared}/{n_total} baseline words kept)")

    # Final summary
    final_sim = cos_sim(v_baseline, words_to_vector(tw[topic_years[-1]], vi))
    final_drift = 1.0 - final_sim
    kept = sorted(set(tw[t0]) & set(tw[topic_years[-1]]))
    lost = sorted(set(tw[t0]) - set(tw[topic_years[-1]]))

    print(f"\n  Total drift ({t0}→{topic_years[-1]}): {final_drift:.4f}")
    print(f"  Baseline words kept: {', '.join(kept) if kept else '(none)'}")
    print(f"  Baseline words lost: {', '.join(lost) if lost else '(none)'}")


### Usage
```python
show_topic_yoy(topic_id=5, subject="cs")
show_topic_endpoint(topic_id=5, subject="cs")
```

In [8]:
# Example: inspect a specific topic
show_topic_yoy(topic_id=0, subject="cs")
show_topic_trajectory(topic_id=0, subject="cs")
show_topic_endpoint(topic_id=0, subject="cs")


YoY Drift: Topic 0 — CS
Active: 2002–2025 (24 years)
  2002→2003: sim=0.0000  drift=1.0000  ██████████████████████████████  🔴 Extreme
           − delaunay, fur, hair, hairstyle, interpolation, makeover, method, moebius, patches, photorealistic
           + agents, animated, canned, dialogues, generator, mnlg, module, neca, pragmatic, pragmatics
  2003→2004: sim=0.0000  drift=1.0000  ██████████████████████████████  🔴 Extreme
           − agents, animated, canned, dialogues, generator, mnlg, module, neca, pragmatic, pragmatics
           + color, colors, edges, graphics, haptic, images, paintings, printouts, substantiated, tactile
  2004→2005: sim=0.0000  drift=1.0000  ██████████████████████████████  🔴 Extreme
           − color, colors, edges, graphics, haptic, images, paintings, printouts, substantiated, tactile
           + content, controvercial, imafe, image, information, mlp, pnn, resolution, sequence, superresolution
  2005→2006: sim=0.1000  drift=0.9000  ███████████████████████